# 영화 관객수 예측 – 부스팅 모델 (LightGBM · XGBoost · CatBoost)

## 이 노트북은 무엇을 하나요?
한 문장으로 요약하면 이렇습니다.
> **"영화 관객수를 맞히는 방법을 여러 가지로 바꿔가며 다 시도해 보고, 어떤 방법이 제일 잘 맞히는지 점수로 비교한다."**


### 우리 프로젝트에서 바꾸는 4가지
| 축 | 선택지 | 라면 비유 |
|---|---|---|
| ① 모델 | LightGBM / XGBoost / CatBoost | 냄비 |
| ② 타깃(맞힐 값) | 관객수 그대로(raw) / log 변환(log) | 물 양 |
| ③ `dir_prev_bfnum` | 열 삭제(drop) / 빈칸을 0으로 채움(fill0) | 계란 |
| ④ 배급사 | 상위 5개+기타(top5) / 관객수 순위로 숫자화(rank) | 파 |

→ 3 × 2 × 2 × 2 = **24개 조합**
→ 여기에 모델마다 1등 조합을 골라 **하이퍼파라미터 튜닝**을 한 번씩 더 해서 3개 추가
→ **제출 파일 총 27개**와 결과 정리표 1개가 만들어집니다.

### 실행 방법
1. Colab 왼쪽 파일 탭에 `movies_train.csv`, `movies_test.csv`, `submission.csv`를 올립니다.
2. 맨 위 셀부터 순서대로 실행합니다(`런타임 → 모두 실행`). 몇 분 정도 걸립니다.
3. 생성된 csv 중 **로컬 점수(CV_RMSE)가 낮은 것부터** 데이콘에 제출합니다.
4. 리더보드 점수를 `부스팅_결과.csv`의 `LB_점수` 칸에 적어 조원들과 공유합니다.

---
## 0. 라이브러리 & 데이터 불러오기

- `lightgbm`, `xgboost`, `catboost`는 세 가지 부스팅 모델 라이브러리입니다.
- Colab에는 lightgbm과 xgboost가 기본으로 설치돼 있지만, **catboost는 설치가 필요할 수 있습니다.** 아래 `!pip install` 줄이 그 역할을 합니다.

**부스팅(Boosting)이란?**
작은 예측기(결정트리)를 하나 만들고 → 그 예측기가 틀린 부분을 다음 예측기가 고치고 → 또 틀린 부분을 다음 예측기가 고치는 식으로, **여러 예측기를 순서대로 쌓아 점점 정확해지는** 방법입니다.
베이스라인의 랜덤포레스트는 예측기를 *동시에 여러 개* 만들어 평균을 내는 방식이라 서로 다릅니다.

| 모델 | 특징 |
|---|---|
| LightGBM | 마이크로소프트가 만듦. 빠르고 가벼움 |
| XGBoost | 가장 오래되고 유명한 부스팅. 대회에서 많이 쓰임 |
| CatBoost | 얀덱스가 만듦. 범주형 데이터(장르·배급사 같은 글자 데이터)에 강하고, 기본 설정만으로도 성능이 좋은 편 |

In [ ]:
!pip install -q catboost   # Colab에 catboost가 없을 때 설치 (이미 있으면 그냥 넘어감)

import itertools, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterGrid   # 교차검증, 파라미터 조합 만들기
from sklearn.metrics import mean_squared_error             # RMSE 계산용
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
warnings.filterwarnings('ignore')   # 경고 메시지 숨기기 (결과에는 영향 없음)

SEED = 42   # 랜덤 시드 고정: 몇 번을 다시 돌려도 같은 결과가 나오게 함

train_raw = pd.read_csv('/content/movies_train.csv')     # 정답(관객수)이 있는 학습용 데이터 600개
test_raw  = pd.read_csv('/content/movies_test.csv')      # 정답이 없는, 우리가 맞혀야 할 데이터 243개
submission = pd.read_csv('/content/submission.csv')      # 제출 양식
print(train_raw.shape, test_raw.shape, submission.shape)

(600, 12) (243, 11) (243, 2)


---
## 1. 전처리 함수 – 데이터를 모델이 먹을 수 있게 바꾸기

모델은 **숫자만** 계산할 수 있어서, 글자 데이터는 숫자로 바꾸고 빈칸은 처리해야 합니다.
베이스라인에서 하던 전처리를 `make_data()`라는 함수 하나로 묶고, **옵션만 바꾸면 다른 버전의 데이터가 나오게** 만들었습니다.

### 모든 버전에서 똑같이 하는 것 (베이스라인과 동일)
- `title`(제목), `director`(감독 이름) 삭제 → 제목은 의미가 없고, 감독은 종류가 너무 많아서
- `release_time`(개봉일)에서 **년, 월**만 뽑아 숫자 변수로 만들기
- 장르·상영등급 같은 글자는 **원핫 인코딩**(`get_dummies`)
  - 예: 장르가 '액션'이면 `genre_액션=1`, 나머지 장르 열은 0

### 옵션 ③ `bfnum` – 감독의 이전 영화 평균 관객수(`dir_prev_bfnum`)를 어떻게 할까?
이 열은 600개 중 **330개가 비어 있습니다.** 비어 있는 건 대부분 그 감독의 첫 작품이라 이전 기록이 없기 때문입니다.
- `'drop'` : 베이스라인처럼 **열을 통째로 삭제**
- `'fill0'` : 빈칸을 **"이전 관객 0명"으로 채워서 살림.** 감독의 흥행 기록은 꽤 쓸모 있는 힌트일 수 있으니까요. 값 차이가 너무 커서 log 변환도 함께 해 줍니다.

### 옵션 ④ `dist` – 배급사(`distributor`)를 어떻게 숫자로 바꿀까?
배급사는 169개나 되어 그대로 원핫 인코딩하면 열이 너무 많아집니다.
- `'top5'` : 베이스라인처럼 **큰 배급사 5개만 남기고 나머지는 '기타'**로 묶은 뒤 원핫 인코딩
- `'rank'` : 배급사마다 **평소 관객수(중앙값) 순위**를 매겨 숫자 하나로 바꿈. CJ처럼 흥행을 잘 시키는 배급사는 높은 숫자를 받습니다. test에만 있는 처음 보는 배급사는 0.

### 베이스라인에서 고친 점
- 베이스라인은 train과 test를 **따로** `get_dummies` 해서, test에 없는 장르가 있으면 열 개수가 달라져 예측할 때 에러가 날 수 있었습니다. → test 열을 train 열에 맞춰 정렬(`reindex`)했습니다.
- XGBoost는 열 이름에 특수문자가 있으면 거부하는 경우가 있어서, 열 이름을 `f0, f1, …`로 바꾸고 원래 이름은 `names`에 따로 보관합니다.

In [ ]:
def make_data(bfnum='drop', dist='top5'):
    tr, te = train_raw.copy(), test_raw.copy()   # 원본은 건드리지 않도록 복사본 사용
    y = tr.pop('box_off_num')                    # 정답(관객수)만 따로 떼어 둠

    # ── 모든 버전 공통 ──
    for df in (tr, te):
        df.drop(['title', 'director'], axis=1, inplace=True)
        df['년'] = df['release_time'].str[:4].astype(int)     # '2012-11-22' → 2012
        df['월'] = df['release_time'].str[5:7].astype(int)    # '2012-11-22' → 11
        df.drop('release_time', axis=1, inplace=True)

    # ── 옵션 ③ dir_prev_bfnum ──
    if bfnum == 'drop':
        tr.drop('dir_prev_bfnum', axis=1, inplace=True)
        te.drop('dir_prev_bfnum', axis=1, inplace=True)
    else:  # 'fill0'
        for df in (tr, te):
            df['dir_prev_bfnum'] = np.log1p(df['dir_prev_bfnum'].fillna(0))

    # ── 옵션 ④ 배급사 ──
    if dist == 'top5':
        top5 = tr['distributor'].value_counts().index[:5]      # 영화 수가 많은 배급사 5개
        for df in (tr, te):
            df['distributor'] = df['distributor'].where(df['distributor'].isin(top5), '기타')
    else:  # 'rank'
        rank = y.groupby(tr['distributor']).median().rank()    # 배급사별 관객수 중앙값의 순위
        for df in (tr, te):
            df['distributor'] = df['distributor'].map(rank).fillna(0)

    # ── 원핫 인코딩 + test 열을 train에 맞추기 ──
    tr = pd.get_dummies(tr, dtype=int)
    te = pd.get_dummies(te, dtype=int).reindex(columns=tr.columns, fill_value=0)

    names = list(tr.columns)                                     # 원래 열 이름 보관 (변수 중요도에 사용)
    tr.columns = te.columns = [f'f{i}' for i in range(tr.shape[1])]
    return tr, y, te, names

# 잘 되는지 한 번 확인해 보기
X, y, X_te, names = make_data('fill0', 'rank')
print('train 모양:', X.shape, '/ test 모양:', X_te.shape)
print(names)

train 모양: (600, 24) / test 모양: (243, 24)
['distributor', 'time', 'dir_prev_bfnum', 'dir_prev_num', 'num_staff', 'num_actor', '년', '월', 'genre_SF', 'genre_공포', 'genre_느와르', 'genre_다큐멘터리', 'genre_드라마', 'genre_멜로/로맨스', 'genre_뮤지컬', 'genre_미스터리', 'genre_서스펜스', 'genre_애니메이션', 'genre_액션', 'genre_코미디', 'screening_rat_12세 관람가', 'screening_rat_15세 관람가', 'screening_rat_전체 관람가', 'screening_rat_청소년 관람불가']


---
## 2. 모델 만들기 & 점수 매기기 함수

### `get_model()` – 이름을 넣으면 모델을 만들어 줌
`'lgbm'`, `'xgb'`, `'cat'` 중 하나를 넣으면 해당 모델을 돌려줍니다. 파라미터를 따로 안 주면 **기본값**으로 만듭니다.

### `run()` – 한 조합을 돌려서 점수와 예측값을 돌려줌
**옵션 ② 타깃(맞힐 값) 처리**
관객수는 1,000명짜리 영화와 1,000만 명짜리 영화가 섞여 있어 차이가 엄청 큽니다.
- `'raw'` : 관객수를 그대로 맞히게 함
- `'log'` : `log1p`로 숫자 범위를 줄여서 맞히게 함 (1,000 → 약 6.9, 1,000만 → 약 16.1). 범위가 줄면 모델이 배우기 쉬워질 수 있습니다. 예측이 끝나면 `expm1`로 **다시 원래 관객수로 되돌립니다.**

**로컬 점수 = 5-Fold 교차검증 RMSE**
리더보드에 내기 전에 내 컴퓨터에서 미리 점수를 매겨 보는 방법입니다.
1. train 600개를 5조각으로 나눕니다.
2. 4조각(480개)으로 공부하고, 남은 1조각(120개)으로 시험을 봅니다.
3. 시험 보는 조각을 바꿔 가며 5번 반복 → 600개 전부에 대해 "모델이 처음 본 상태에서의 예측값"이 생깁니다.
4. 이 예측값과 실제 관객수로 **RMSE**를 계산합니다.

**RMSE(Root Mean Squared Error)** = 대략 "평균적으로 몇 명 정도 틀렸는지". **낮을수록 좋습니다.** 데이콘 리더보드도 RMSE로 점수를 매깁니다.

점수를 다 매긴 뒤에는 **train 600개 전부로 다시 학습**해서 test를 예측합니다(공부 자료가 많을수록 좋으니까요). 관객수가 음수로 나오면 말이 안 되니 0 미만은 0으로 고칩니다.

In [ ]:
def get_model(name, params=None):
    params = params or {}
    if name == 'lgbm':
        return LGBMRegressor(random_state=SEED, verbose=-1, **params)   # verbose=-1: 학습 로그 숨김
    if name == 'xgb':
        return XGBRegressor(random_state=SEED, **params)
    return CatBoostRegressor(random_seed=SEED, verbose=0, **params)     # verbose=0: 학습 로그 숨김

def run(model_name, target, X, y, X_te, params=None, n_splits=5):
    # 타깃 변환: log면 log1p로 줄이고, 예측 후 expm1로 되돌림
    y_fit = np.log1p(y) if target == 'log' else y
    inv = np.expm1 if target == 'log' else (lambda p: p)

    # ── 5-Fold 교차검증 (모의고사) ──
    oof = np.zeros(len(y))   # oof = out-of-fold, "처음 본 상태에서의 예측값"을 담을 곳
    for tr_idx, va_idx in KFold(n_splits, shuffle=True, random_state=SEED).split(X):
        m = get_model(model_name, params)
        m.fit(X.iloc[tr_idx], y_fit.iloc[tr_idx])          # 4조각으로 공부
        oof[va_idx] = inv(m.predict(X.iloc[va_idx]))       # 1조각으로 시험
    rmse = mean_squared_error(y, oof) ** 0.5               # 평균적으로 몇 명 틀렸나

    # ── 전체 데이터로 다시 학습 → test 예측 ──
    m = get_model(model_name, params)
    m.fit(X, y_fit)
    pred = np.clip(inv(m.predict(X_te)), 0, None)          # 음수 관객수는 0으로
    return rmse, pred, m

---
## 3. 기본 조합 24개 전부 돌리기

`itertools.product`가 4가지 옵션의 **모든 조합**을 만들어 줍니다. (3 × 2 × 2 × 2 = 24개)
조합마다 ① 데이터 만들기 → ② 모델 학습 & 로컬 점수 계산 → ③ 제출 파일 저장을 반복합니다.

**파일 이름이 곧 조합 이름입니다.** 예: `cat_log_fill0_rank.csv`
= CatBoost 모델 + log 타깃 + dir_prev_bfnum을 0으로 채움 + 배급사 순위 인코딩

CatBoost는 다른 두 모델보다 학습이 조금 느려서, 이 셀은 몇 분 정도 걸릴 수 있습니다.

In [ ]:
MODELS  = ['lgbm', 'xgb', 'cat']   # ① 모델
TARGETS = ['raw', 'log']           # ② 타깃
BFNUMS  = ['drop', 'fill0']        # ③ dir_prev_bfnum 처리
DISTS   = ['top5', 'rank']         # ④ 배급사 처리

results = []
for model_name, target, bfnum, dist in itertools.product(MODELS, TARGETS, BFNUMS, DISTS):
    X, y, X_te, _ = make_data(bfnum, dist)                    # 이 조합용 데이터 만들기
    rmse, pred, _ = run(model_name, target, X, y, X_te)       # 학습 + 로컬 점수 + 예측

    fname = f'{model_name}_{target}_{bfnum}_{dist}.csv'       # 조합 이름 = 파일 이름
    sub = submission.copy()
    sub['box_off_num'] = pred
    sub.to_csv(fname, index=False)                            # 제출 파일 저장

    results.append({'파일': fname, '모델': model_name, '타깃': target, 'bfnum': bfnum,
                    '배급사': dist, 'CV_RMSE': round(rmse)})
    print(f'{fname:35s} 로컬 RMSE = {rmse:,.0f}')

# 로컬 점수가 좋은(낮은) 순서로 정렬해서 보기
res = pd.DataFrame(results).sort_values('CV_RMSE').reset_index(drop=True)
res

lgbm_raw_drop_top5.csv              로컬 RMSE = 1,561,201
lgbm_raw_drop_rank.csv              로컬 RMSE = 1,538,100
lgbm_raw_fill0_top5.csv             로컬 RMSE = 1,520,669
lgbm_raw_fill0_rank.csv             로컬 RMSE = 1,517,725
lgbm_log_drop_top5.csv              로컬 RMSE = 1,513,912
lgbm_log_drop_rank.csv              로컬 RMSE = 1,521,881
lgbm_log_fill0_top5.csv             로컬 RMSE = 1,573,954
lgbm_log_fill0_rank.csv             로컬 RMSE = 1,557,661
xgb_raw_drop_top5.csv               로컬 RMSE = 1,651,184
xgb_raw_drop_rank.csv               로컬 RMSE = 1,629,874
xgb_raw_fill0_top5.csv              로컬 RMSE = 1,693,040
xgb_raw_fill0_rank.csv              로컬 RMSE = 1,577,008
xgb_log_drop_top5.csv               로컬 RMSE = 1,745,595
xgb_log_drop_rank.csv               로컬 RMSE = 1,573,593
xgb_log_fill0_top5.csv              로컬 RMSE = 1,603,595
xgb_log_fill0_rank.csv              로컬 RMSE = 1,500,884
cat_raw_drop_top5.csv               로컬 RMSE = 1,479,568
cat_raw_drop_rank.csv               로컬 RMSE = 1,

,파일,모델,타깃,bfnum,배급사,CV_RMSE
0,cat_log_drop_rank.csv,cat,log,drop,rank,1430470
1,cat_raw_drop_rank.csv,cat,raw,drop,rank,1458274
2,cat_log_drop_top5.csv,cat,log,drop,top5,1465920
3,cat_raw_fill0_rank.csv,cat,raw,fill0,rank,1471851
4,cat_log_fill0_rank.csv,cat,log,fill0,rank,1478615
5,cat_raw_drop_top5.csv,cat,raw,drop,top5,1479568
6,cat_log_fill0_top5.csv,cat,log,fill0,top5,1487244
7,cat_raw_fill0_top5.csv,cat,raw,fill0,top5,1488925
8,xgb_log_fill0_rank.csv,xgb,log,fill0,rank,1500884
9,lgbm_log_drop_top5.csv,lgbm,log,drop,top5,1513912


---
## 4. 하이퍼파라미터 튜닝 (모델별 1등 조합으로)

**하이퍼파라미터**는 모델의 세부 설정입니다. 라면으로 치면 "불 세기, 끓이는 시간" 같은 것이고, 3번에서는 전부 기본값을 썼습니다.
여기서는 3번에서 **모델별로 로컬 점수가 가장 좋았던 조합**을 하나씩 골라, 설정값을 여러 개 바꿔 보며 더 좋은 설정을 찾습니다.

| 설정 | 뜻 |
|---|---|
| `n_estimators` / `iterations` | 예측기(나무)를 몇 개 쌓을지. 많을수록 꼼꼼하지만 너무 많으면 외워 버림(과적합) |
| `learning_rate` | 한 번에 얼마나 크게 고칠지. 작으면 천천히 꼼꼼하게, 크면 빠르게 |
| `num_leaves` / `max_depth` / `depth` | 나무 하나가 얼마나 복잡한지. 데이터가 600개뿐이라 너무 복잡하면 과적합 |
| `min_child_samples` | 나무의 한 가지 끝에 최소 몇 개의 영화가 있어야 하는지 |
| `subsample` | 나무 하나를 만들 때 데이터를 몇 %만 쓸지 (일부만 쓰면 과적합이 줄어듦) |
| `l2_leaf_reg` | CatBoost의 과적합 방지 강도 |

`ParameterGrid`가 아래 후보의 모든 조합(모델당 2×2×2×2 = 16개)을 만들어 줍니다. 모두 돌려 본 뒤 로컬 점수가 가장 좋은 설정으로 제출 파일을 만듭니다(파일 이름 끝에 `_tuned`).

※ 튜닝 후 점수가 오히려 조금 나빠질 수도 있습니다. 기본값이 이미 잘 맞았다는 뜻이니 이상한 게 아닙니다.

In [ ]:
grids = {
    'lgbm': {'n_estimators': [300, 1000], 'learning_rate': [0.01, 0.05],
             'num_leaves': [7, 31], 'min_child_samples': [5, 20]},
    'xgb':  {'n_estimators': [300, 1000], 'learning_rate': [0.01, 0.05],
             'max_depth': [3, 6], 'subsample': [0.8, 1.0]},
    'cat':  {'iterations': [300, 1000], 'learning_rate': [0.03, 0.1],
             'depth': [4, 6], 'l2_leaf_reg': [1, 3]},
}

tuned = []
for model_name in MODELS:
    best = res[res['모델'] == model_name].iloc[0]             # 이 모델의 1등 조합
    X, y, X_te, _ = make_data(best['bfnum'], best['배급사'])

    best_rmse, best_params, best_pred = np.inf, None, None
    for params in ParameterGrid(grids[model_name]):           # 설정 16개를 하나씩 시험
        rmse, pred, _ = run(model_name, best['타깃'], X, y, X_te, params)
        if rmse < best_rmse:                                  # 더 좋으면 기록 갱신
            best_rmse, best_params, best_pred = rmse, params, pred

    fname = f"{model_name}_{best['타깃']}_{best['bfnum']}_{best['배급사']}_tuned.csv"
    sub = submission.copy()
    sub['box_off_num'] = best_pred
    sub.to_csv(fname, index=False)

    tuned.append({'파일': fname, '모델': model_name, '타깃': best['타깃'], 'bfnum': best['bfnum'],
                  '배급사': best['배급사'], 'CV_RMSE': round(best_rmse), '파라미터': str(best_params)})
    print(f'{model_name}: 튜닝 전 {best["CV_RMSE"]:,} → 튜닝 후 {best_rmse:,.0f}  {best_params}')

lgbm: 튜닝 전 1,513,912 → 튜닝 후 1,490,659  {'learning_rate': 0.01, 'min_child_samples': 20, 'n_estimators': 1000, 'num_leaves': 7}
xgb: 튜닝 전 1,500,884 → 튜닝 후 1,434,137  {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}
cat: 튜닝 전 1,430,470 → 튜닝 후 1,426,360  {'depth': 6, 'iterations': 1000, 'l2_leaf_reg': 3, 'learning_rate': 0.03}


---
## 5. 전체 결과표 저장 (조원 공유용)

27개 결과를 로컬 점수 순으로 정리해 `부스팅_결과.csv`로 저장합니다.
- `LB_점수` 칸은 비어 있습니다. **데이콘에 제출한 뒤 리더보드 점수를 직접 채워 넣으세요.**
- 하루 제출 횟수 제한이 있다면 **위에서부터(로컬 점수가 좋은 것부터)** 제출하면 됩니다.
- 로컬 점수와 리더보드 점수의 순위가 꼭 같지는 않습니다. 데이터가 적어서 생기는 자연스러운 차이이고, 이 차이도 발표 거리가 됩니다.
- `utf-8-sig`로 저장해서 엑셀로 열어도 한글이 깨지지 않습니다.

In [ ]:
final = pd.concat([res, pd.DataFrame(tuned)], ignore_index=True)
final['파라미터'] = final['파라미터'].fillna('기본값')
final = final.sort_values('CV_RMSE').reset_index(drop=True)
final['LB_점수'] = ''
final.to_csv('부스팅_결과.csv', index=False, encoding='utf-8-sig')
final

,파일,모델,타깃,bfnum,배급사,CV_RMSE,파라미터,LB_점수
0,cat_log_drop_rank_tuned.csv,cat,log,drop,rank,1426360,"{'depth': 6, 'iterations': 1000, 'l2_leaf_reg'...",
1,cat_log_drop_rank.csv,cat,log,drop,rank,1430470,기본값,
2,xgb_log_fill0_rank_tuned.csv,xgb,log,fill0,rank,1434137,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",
3,cat_raw_drop_rank.csv,cat,raw,drop,rank,1458274,기본값,
4,cat_log_drop_top5.csv,cat,log,drop,top5,1465920,기본값,
5,cat_raw_fill0_rank.csv,cat,raw,fill0,rank,1471851,기본값,
6,cat_log_fill0_rank.csv,cat,log,fill0,rank,1478615,기본값,
7,cat_raw_drop_top5.csv,cat,raw,drop,top5,1479568,기본값,
8,cat_log_fill0_top5.csv,cat,log,fill0,top5,1487244,기본값,
9,cat_raw_fill0_top5.csv,cat,raw,fill0,top5,1488925,기본값,


---
##6. (선택) 변수 중요도 – 발표 자료용

로컬 점수 1등 모델이 예측할 때 **어떤 정보를 많이 참고했는지** 보여 줍니다.
숫자가 클수록 그 변수가 관객수를 맞히는 데 중요했다는 뜻입니다. 예를 들어 `num_staff`(스태프 수)가 위에 있다면 "스태프가 많은 큰 영화일수록 관객이 많다"는 패턴을 모델이 찾았다는 의미입니다.

(모델마다 중요도 계산 방식이 달라서 숫자의 크기 자체는 모델끼리 비교하지 않습니다.)

In [ ]:
top = final.iloc[0]   # 로컬 점수 1등
X, y, X_te, names = make_data(top['bfnum'], top['배급사'])
params = eval(top['파라미터']) if top['파라미터'] != '기본값' else None
_, _, model = run(top['모델'], top['타깃'], X, y, X_te, params)

imp = pd.Series(model.feature_importances_, index=names).sort_values(ascending=False)
print('1등 조합:', top['파일'])
imp.head(15)

1등 조합: cat_log_drop_rank_tuned.csv


,0
distributor,43.865502
num_staff,15.812986
time,10.574200
월,7.445127
년,4.997852
num_actor,4.458689
dir_prev_num,3.195591
genre_드라마,1.938270
screening_rat_전체 관람가,1.774739
screening_rat_청소년 관람불가,1.621387


---
## 7. 베이스라인(랜덤포레스트) 점수 – 비교 기준선

올려 주신 베이스라인 코드는 **랜덤포레스트**를 썼습니다. 부스팅이 얼마나 좋아졌는지 보려면 베이스라인과 **똑같은 조건**(랜덤포레스트, `dir_prev_bfnum` 삭제, 배급사 top5, 관객수 그대로)으로 점수를 재야 합니다.

- 랜덤포레스트: 나무 여러 개를 **따로따로 동시에** 만들어 평균 (여러 명에게 답을 받아 평균)
- 부스팅: 나무를 **순서대로** 만들며 앞 나무가 틀린 부분을 고침 (오답 노트)

원래 베이스라인은 `random_state`를 정하지 않아 돌릴 때마다 점수가 조금씩 달랐는데, 여기서는 시드를 고정했습니다. 리더보드 점수를 보려면 `rf_baseline.csv`를 데이콘에 제출하세요.

In [ ]:
## 베이스라인

from sklearn.ensemble import RandomForestRegressor

# 베이스라인과 같은 전처리: dir_prev_bfnum 삭제, 배급사 상위5+기타
X, y, X_te, _ = make_data('drop', 'top5')

# 5-Fold 교차검증으로 로컬 점수(RMSE) 계산
oof = np.zeros(len(y))
for tr_idx, va_idx in KFold(5, shuffle=True, random_state=SEED).split(X):
    m = RandomForestRegressor(n_estimators=100, random_state=SEED)   # 베이스라인과 같은 모델
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])                             # 관객수 그대로(raw) 학습
    oof[va_idx] = m.predict(X.iloc[va_idx])
base_rmse = mean_squared_error(y, oof) ** 0.5

print(f'베이스라인(랜덤포레스트) 로컬 RMSE = {base_rmse:,.0f}')
print(f'최고 부스팅 대비 개선율 = {(base_rmse - final.iloc[0]["CV_RMSE"]) / base_rmse:.1%}')

베이스라인(랜덤포레스트) 로컬 RMSE = 1,516,505
최고 부스팅 대비 개선율 = 5.9%


In [ ]:
# 전체 train으로 다시 학습해서 test 예측 → 제출 파일 저장
m = RandomForestRegressor(n_estimators=100, random_state=SEED)
m.fit(X, y)
sub = submission.copy()
sub['box_off_num'] = m.predict(X_te)
sub.to_csv('rf_baseline.csv', index=False)

---
## 8. 최종: 시드 평균 + 3모델 앙상블 (리더보드 결과 반영)

### 지금까지의 리더보드 결과
| 순위 | 파일 | 리더보드 RMSE | 베이스라인 대비 |
|---|---|---|---|
| 1 | `cat_raw_fill0_rank` | 1,277,640 | 9.9% 개선 |
| 2 | `xgb_log_fill0_rank_tuned` | 1,286,295 | 9.3% 개선 |
| 3 | `cat_log_fill0_rank` | 1,306,681 | 7.8% 개선 |
| 4 | `lgbm_raw_fill0_rank` | 1,320,540 | 6.8% 개선 |
| 5 | `xgb_log_fill0_rank` | 1,366,653 | 3.6% 개선 |
| 6 | `cat_log_drop_rank_tuned` | 1,392,003 | 1.8% 개선 |
| 7 | `rf_baseline` (기준선) | 1,417,428 | – |
| 8 | `lgbm_log_drop_top5_tuned` | 1,490,264 | 5.1% 나빠짐 |

→ **`fill0 + rank` 전처리가 상위 1~5위를 모두 차지**했습니다. 모델이나 튜닝보다 전처리가 점수를 가장 크게 바꿨습니다.

### 이 셀이 하는 일 (비유: 시험 잘 본 학생 3명의 답을 모으기)
1. **데이터:** 상위권이 모두 쓴 `fill0 + rank` 전처리 사용
2. **설정:** 리더보드에서 **실제로 검증된** 모델별 최고 파일의 설정을 그대로 사용
   - XGBoost: 튜닝 설정 + log 타깃 (2위 파일)
   - CatBoost: 기본 설정 + raw 타깃 (1위 파일)
   - LightGBM: 기본 설정 + raw 타깃 (4위 파일)
3. **시드 평균:** 같은 모델을 랜덤값(시드)만 바꿔 5번 학습하고 평균 → 운에 덜 흔들림 (한 학생에게 5번 풀게 하고 평균)
4. **앙상블:** 세 모델의 예측을 **1:1:1**로 평균 → 서로 틀리는 영화가 달라서 실수를 덮어 줌
   - 로컬 점수로 '최적 비율'을 찾지 않은 이유: CatBoost의 drop/fill0 비교에서 로컬 점수가 리더보드와 반대로 나온 적이 있어서, 가장 단순하고 안전한 1:1:1을 썼습니다.

결과 파일: `ens3_seed_fill0_rank.csv` → 제출 후 1,277,640보다 낮으면(좋으면) 이 파일을 최종으로 선택하세요.

In [ ]:
# 1) 상위권 전처리
X, y, X_te, _ = make_data('fill0', 'rank')

# 2) 리더보드에서 검증된 설정
xgb_tuned = [t for t in tuned if t['모델'] == 'xgb'][0]
assert (xgb_tuned['bfnum'], xgb_tuned['배급사']) == ('fill0', 'rank'), 'xgb 튜닝 조합이 fill0_rank가 아닙니다 → 아래 빠른 버전을 쓰세요'
configs = {
    'xgb':  {'target': 'log', 'params': eval(xgb_tuned['파라미터'])},  # xgb_log_fill0_rank_tuned (1,286,295)
    'cat':  {'target': 'raw', 'params': {}},                            # cat_raw_fill0_rank       (1,277,640) 기본값
    'lgbm': {'target': 'raw', 'params': {}},                            # lgbm_raw_fill0_rank      (1,320,540) 기본값
}
print('사용할 설정:', configs)

def make_model(name, params, seed):
    if name == 'lgbm':
        return LGBMRegressor(random_state=seed, verbose=-1, **params)
    if name == 'xgb':
        return XGBRegressor(random_state=seed, **params)
    return CatBoostRegressor(random_seed=seed, verbose=0, **params)

def run_seeds(name, target, params, seeds=(0, 1, 2, 3, 4)):
    """시드 5개로 학습한 결과를 평균"""
    y_fit = np.log1p(y) if target == 'log' else y
    inv = np.expm1 if target == 'log' else (lambda p: p)
    oof = np.zeros(len(y))
    pred = np.zeros(len(X_te))
    for s in seeds:
        for tr_idx, va_idx in KFold(5, shuffle=True, random_state=SEED).split(X):   # 모의고사
            m = make_model(name, params, s)
            m.fit(X.iloc[tr_idx], y_fit.iloc[tr_idx])
            oof[va_idx] += inv(m.predict(X.iloc[va_idx])) / len(seeds)
        m = make_model(name, params, s)                                              # 전체로 학습 → test 예측
        m.fit(X, y_fit)
        pred += inv(m.predict(X_te)) / len(seeds)
    return oof, np.clip(pred, 0, None)

rmse = lambda p: mean_squared_error(y, p) ** 0.5

# 3) 모델별 시드 평균 예측 (몇 분 걸림)
oofs, preds = {}, {}
for name, cfg in configs.items():
    oofs[name], preds[name] = run_seeds(name, cfg['target'], cfg['params'])
    print(f'{name}: 시드 5개 평균 로컬 RMSE = {rmse(oofs[name]):,.0f}')

# 4) 1:1:1 앙상블 → 저장
print(f'\n3모델 1:1:1 앙상블 로컬 RMSE = {rmse(sum(oofs.values()) / 3):,.0f}  (참고용)')
sub = submission.copy()
sub['box_off_num'] = sum(preds.values()) / 3
sub.to_csv('ens3_seed_fill0_rank.csv', index=False)
print('저장 완료: ens3_seed_fill0_rank.csv')

사용할 설정: {'xgb': {'target': 'log', 'params': {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}}, 'cat': {'target': 'raw', 'params': {}}, 'lgbm': {'target': 'raw', 'params': {}}}
xgb: 시드 5개 평균 로컬 RMSE = 1,434,137
cat: 시드 5개 평균 로컬 RMSE = 1,480,744
lgbm: 시드 5개 평균 로컬 RMSE = 1,517,725

3모델 1:1:1 앙상블 로컬 RMSE = 1,417,218  (참고용)
저장 완료: ens3_seed_fill0_rank.csv


---
## 📌 Summary: 제출 파일을 고른 기준과 앙상블까지 온 과정

### 한 줄 결론
`fill0 + rank` 전처리를 적용한 부스팅 3종(XGBoost · CatBoost · LightGBM)을 1:1:1로 앙상블해서
**리더보드 RMSE 1,232,606**을 받았습니다. 베이스라인(랜덤포레스트, 1,417,428)보다 **13.0% 개선**된 점수입니다.

### 진행 과정 (총 9번 제출)

**1단계 · 24개 조합을 로컬 점수로 실험 (제출 전)**
- 모델 3종 × 타깃(raw/log) × `dir_prev_bfnum`(drop/fill0) × 배급사(top5/rank) = 24개 조합을 5-Fold CV로 비교했습니다.
- 모델마다 1등 조합으로 하이퍼파라미터 튜닝을 했습니다.
- 26개 파일을 전부 제출하기는 어려워서, **로컬 점수로 후보를 좁힌 뒤 리더보드로 확인**하기로 했습니다.

**2단계 · 1차 제출: 모델별 로컬 1등 + 베이스라인 (4개)**
- 기준: 모델마다 로컬 점수가 가장 좋았던 튜닝 파일 1개씩, 그리고 비교 기준이 될 베이스라인을 냈습니다.
- 결과:
  - XGBoost(`fill0+rank`)가 1등이었습니다. 로컬 1등이던 CatBoost는 2등으로 밀려서, **로컬 순위와 리더보드 순위가 다를 수 있다**는 걸 확인했습니다.
  - 배급사를 **rank**로 처리한 파일은 베이스라인을 이겼고, **top5**로 처리한 LightGBM은 오히려 베이스라인에 졌습니다.
- 여기서 세운 가설: **"모델보다 전처리(특히 배급사 rank)가 중요하다."**

**3단계 · 2차 제출: 가설 확인용 (3개)**
제출한 파일마다 확인하려는 게 하나씩 있었습니다.
| 제출 파일 | 확인하려던 것 | 결과 |
|---|---|---|
| `lgbm_raw_fill0_rank` | LightGBM도 rank로 바꾸면 베이스라인을 이길까? | 이김 (6.8% 개선) → rank 효과 확인 |
| `cat_log_fill0_rank` | XGBoost가 이긴 게 모델 덕일까, `fill0` 전처리 덕일까? | CatBoost도 크게 좋아짐 → `fill0` 효과 확인 |
| `xgb_log_fill0_rank` (튜닝 전) | 튜닝이 리더보드에서도 효과가 있을까? | 튜닝 후 5.9% 개선 → 튜닝 효과 확인 |

→ **`fill0 + rank` 파일이 모델과 상관없이 상위권을 모두 차지**했습니다.

**4단계 · 3차 제출: 남은 유력 후보 (1개)**
- `fill0 + rank` 조합 중 아직 제출하지 않은 파일은 3개였습니다. 그중 **로컬 점수가 이미 제출한 짝보다 좋았던 파일**은 `cat_raw_fill0_rank` 하나뿐이어서 이것만 냈습니다.
- 결과: 1,277,640으로 단일 모델 1등이었습니다. CatBoost는 **튜닝 없이 기본값으로도** 가장 좋았습니다.

**5단계 · 앙상블 (최종)**
리더보드 결과를 보고 앙상블 설계를 다음과 같이 정했습니다.
- **전처리:** 상위권을 모두 차지한 `fill0 + rank`를 사용했습니다.
- **설정:** 로컬 점수가 아니라 **리더보드에서 검증된 모델별 최고 파일의 설정**을 그대로 가져왔습니다.
  - XGBoost: 튜닝 설정 + log 타깃
  - CatBoost: 기본값 + raw 타깃
  - LightGBM: 기본값 + raw 타깃
- **시드 평균:** 같은 모델을 시드만 바꿔 5번 학습하고 평균을 냈습니다. 운에 따라 점수가 흔들리는 걸 줄이기 위해서입니다.
- **1:1:1 평균:** 로컬 점수로 섞는 비율을 최적화하지 않았습니다. CatBoost의 drop/fill0 비교에서 로컬 점수가 리더보드와 반대로 나온 적이 있어서, 가장 단순하고 안전한 방법을 택했습니다.
- 결과: **1,232,606.** 단일 최고 모델보다 3.5% 더 좋았습니다. 세 모델이 틀리는 영화가 서로 달라서 실수가 서로 덮인 것으로 보입니다.

### 리더보드 결과 (좋은 순)
| 순위 | 파일 | 리더보드 RMSE | 베이스라인 대비 |
|---|---|---|---|
| 1 | `ens3_seed_fill0_rank` (앙상블) | **1,232,606** | **+13.0%** |
| 2 | `cat_raw_fill0_rank` | 1,277,640 | +9.9% |
| 3 | `xgb_log_fill0_rank_tuned` | 1,286,295 | +9.3% |
| 4 | `cat_log_fill0_rank` | 1,306,681 | +7.8% |
| 5 | `lgbm_raw_fill0_rank` | 1,320,540 | +6.8% |
| 6 | `xgb_log_fill0_rank` | 1,366,653 | +3.6% |
| 7 | `cat_log_drop_rank_tuned` | 1,392,003 | +1.8% |
| 8 | `rf_baseline` (기준선) | 1,417,428 | – |
| 9 | `lgbm_log_drop_top5_tuned` | 1,490,264 | −5.1% |

### 핵심 발견
1. **전처리가 가장 중요했습니다.** `dir_prev_bfnum` 빈칸을 0으로 채우고(빈칸 = 감독 데뷔작) 배급사를 관객수 순위로 바꾸니, 모델과 상관없이 점수가 크게 좋아졌습니다.
2. **log 타깃은 모델마다 달랐습니다.** CatBoost와 LightGBM은 raw가, XGBoost는 log가 더 좋았습니다. log는 대박 영화를 작게 예측하는 경향이 있습니다.
3. **튜닝은 XGBoost에서 효과가 컸고, CatBoost는 기본값으로도 충분했습니다.**
4. **로컬 점수는 참고용이고, 리더보드로 꼭 확인해야 합니다.** train이 600편뿐이라 두 점수의 순위가 다르게 나오는 경우가 있었습니다.
5. **성격이 다른 모델을 섞는 앙상블이 단일 모델보다 좋았습니다.**
